# 08 — Migration Results

Consolidated view across every step: accounts, addresses, and orders.
Reads only from `migration_data/` — safe to re-run any time without
touching OneBill.

## 1. Setup

In [13]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))

from onebill_common import *  # noqa: F401,F403

df_account_results  = try_load_df("account_results")
df_address_results  = try_load_df("address_results")
df_order_results    = try_load_df("order_results")


## 2. Account creation summary

In [14]:
if df_account_results is not None:
    print(df_account_results["status"].value_counts().to_string())
    display(df_account_results)
else:
    print("04_Create_Accounts.ipynb has not been run yet.")


status
created    2


,AccountCode,AccountCode_Batch,AccountName,AccountKey,status,onebill_id,error
0,999656921,999656921_fullbatch,Managed by Williams (99965692_fullbatch1),managed_by_williams,created,unknown,NaN
1,99965692,99965692_fullbatch,Williams Internet Limited (99965692_fullbatch),NaN,created,unknown,NaN


## 3. Address creation summary + failures

In [15]:
if df_address_results is not None:
    print(df_address_results["status"].value_counts().to_string())
    address_failures = df_address_results[df_address_results["status"] != "created"]
    print(f"\n{len(address_failures):,} not created (failed or skipped)")
else:
    address_failures = pd.DataFrame()
    print("06_Create_Addresses.ipynb has not been run yet.")

address_failures.head(20)



status
created    130
exists     126
skipped     67

193 not created (failed or skipped)


,SubscriptionUSN,TargetAccountNumber,addLine1,status,ship_add_id,error
0,V113062897,999656921_fullbatch,NaN,skipped,NaN,circuits lookup failed: 404 Client Error: Not ...
1,V113062905,99965692_fullbatch,NaN,skipped,NaN,circuits lookup failed: 404 Client Error: Not ...
2,V113062335,99965692_fullbatch,NaN,skipped,NaN,no SupplierServiceID on this subscription
3,V113062343,99965692_fullbatch,NaN,skipped,NaN,no SupplierServiceID on this subscription
4,V113062913,99965692_fullbatch,NaN,skipped,NaN,circuits lookup failed: 404 Client Error: Not ...
5,V113062418,99965692_fullbatch,11 Matata Way,exists,141207.0,NaN
6,V113062400,99965692_fullbatch,5 Matata Way,exists,141205.0,NaN
7,V113062962,99965692_fullbatch,18/10 Fathom Place,exists,141206.0,NaN
8,V113061451,99965692_fullbatch,39 Chester Street West,exists,141105.0,NaN
9,V113062988,99965692_fullbatch,3/23 Awaroa Road,exists,141108.0,NaN


In [16]:
no_address = df_address_results[df_address_results["status"].isin(["failed", "skipped"])].copy()

df_subs_lookup = try_load_df("subscriptions_resolved")
if df_subs_lookup is not None:
    no_address = no_address.merge(
        df_subs_lookup[["SubscriptionUSN", "SupplierServiceID"]],
        on="SubscriptionUSN",
        how="left",
    )

out_path = f'Migration_data/Subscriptions_Without_Address_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
no_address.to_csv(out_path, index=False)
print(f"Wrote {len(no_address):,} subscriptions without an address to {out_path}")
no_address.head(20)

Wrote 67 subscriptions without an address to Migration_data/Subscriptions_Without_Address_20260723_103228.csv


,SubscriptionUSN,TargetAccountNumber,addLine1,status,ship_add_id,error,SupplierServiceID
0,V113062897,999656921_fullbatch,NaN,skipped,NaN,circuits lookup failed: 404 Client Error: Not ...,ENVOYB02619432
1,V113062905,99965692_fullbatch,NaN,skipped,NaN,circuits lookup failed: 404 Client Error: Not ...,ENVOYB02619137
2,V113062335,99965692_fullbatch,NaN,skipped,NaN,no SupplierServiceID on this subscription,NaN
3,V113062343,99965692_fullbatch,NaN,skipped,NaN,no SupplierServiceID on this subscription,NaN
4,V113062913,99965692_fullbatch,NaN,skipped,NaN,circuits lookup failed: 404 Client Error: Not ...,ENVOYB02619148
5,V113063077,99965692_fullbatch,NaN,skipped,NaN,circuits lookup failed: 404 Client Error: Not ...,1643175796
6,V113063259,99965692_fullbatch,NaN,skipped,NaN,circuits lookup failed: 404 Client Error: Not ...,1643175967
7,V113063267,99965692_fullbatch,NaN,skipped,NaN,circuits lookup failed: 404 Client Error: Not ...,1643176031
8,V113064125,99965692_fullbatch,NaN,skipped,NaN,circuits lookup failed: 404 Client Error: Not ...,1643180344
9,V113064349,99965692_fullbatch,NaN,skipped,NaN,circuits lookup failed: 404 Client Error: Not ...,1643180612


In [17]:
if not address_failures.empty:
    out_path = f'Address_Failures_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    address_failures.to_csv(out_path, index=False)
    print(f"Wrote {len(address_failures):,} address failures/skips to {out_path}")


Wrote 193 address failures/skips to Address_Failures_20260723_103228.csv


## 4. Order creation summary + failures

In [18]:
if df_order_results is not None:
    print(df_order_results["status"].value_counts().to_string())
    order_failures = df_order_results[df_order_results["status"] == "failed"]
    print(f"\n{len(order_failures):,} failed orders")
else:
    order_failures = pd.DataFrame()
    print("07_Create_Subscription_Orders.ipynb has not been run yet.")

order_failures.head(20)


status
success    246
failed      10

10 failed orders


,SubscriptionUSN,TargetAccountNumber,status,plan_matched,productName,priceplanName,onebill_order_id,error
0,V113061451,99965692_fullbatch,failed,True,Wholesale Fibre BS2 (Enable),WS Tail+Data - BS2 Res (Enable) - 920/500,NaN,Provisioning attribute value ENVOYB02618143 is...
1,V113062392,99965692_fullbatch,failed,True,Wholesale Fibre BS2 (TFF),WS Tail+Data - BS2 Res Fibre Starter (TFF) - 1...,NaN,Provisioning attribute value UFF000007009635 i...
2,V113062384,99965692_fullbatch,failed,True,Wholesale Fibre BS2 (TFF),WS Tail+Data - BS2 Res Fibre Starter (TFF) - 1...,NaN,Provisioning attribute value UFF000007009674 i...
20,V113063150,99965692_fullbatch,failed,True,Wholesale Fibre BS2 (Chorus),WS Tail+Data - BS2 Res Fibre Starter (Chorus) ...,NaN,Provisioning attribute value 1643175882 is alr...
45,V113064323,99965692_fullbatch,failed,True,Wholesale Fibre BS2 (Chorus),WS Tail+Data - BS2 Res Fibre Starter (Chorus) ...,NaN,Provisioning attribute value 1643180415 is alr...
85,V113069066,99965692_fullbatch,failed,True,Wholesale Fibre BS2 (Chorus),WS Tail+Data - BS2 Res (Chorus) - 500/100,NaN,Provisioning attribute value 1643216783 is alr...
111,V113071302,99965692_fullbatch,failed,True,Wholesale Fibre BS2 (Chorus),WS Tail+Data - BS2 Res (Chorus) - 920/500,NaN,Provisioning attribute value 1643230569 is alr...
124,V113073274,99965692_fullbatch,failed,True,Wholesale Fibre BS2 (Chorus),WS Tail+Data - BS2 Res Fibre Starter (Chorus) ...,NaN,Provisioning attribute value 1643243278 is alr...
182,V113080527,999656921_fullbatch,failed,True,Wholesale Fibre BS2 (Enable),WS Tail+Data - BS2 Res Fibre Starter (Enable) ...,NaN,Provisioning attribute value ENVOYB02636755 is...
245,V113091029,999656921_fullbatch,failed,True,Wholesale Fibre BS2 (Enable),WS Tail+Data - BS2 Res Fibre Starter (Enable) ...,NaN,Provisioning attribute value ENVOYB02648625 is...


In [19]:
if not order_failures.empty:
    out_path = f'Order_Failures_{datetime.now().strftime("%Y%m%d_%H%M%S")}.csv'
    order_failures.to_csv(out_path, index=False)
    print(f"Wrote {len(order_failures):,} order failures to {out_path}")

    error_summary = (
        order_failures.groupby("error")
        .agg(count=("SubscriptionUSN", "size"), example=("SubscriptionUSN", "first"))
        .sort_values("count", ascending=False)
        .reset_index()
    )
else:
    error_summary = pd.DataFrame()

error_summary


Wrote 10 order failures to Order_Failures_20260723_103228.csv


,error,count,example
0,Provisioning attribute value 1643175882 is alr...,1,V113063150
1,Provisioning attribute value 1643180415 is alr...,1,V113064323
2,Provisioning attribute value 1643216783 is alr...,1,V113069066
3,Provisioning attribute value 1643230569 is alr...,1,V113071302
4,Provisioning attribute value 1643243278 is alr...,1,V113073274
5,Provisioning attribute value ENVOYB02618143 is...,1,V113061451
6,Provisioning attribute value ENVOYB02636755 is...,1,V113080527
7,Provisioning attribute value ENVOYB02648625 is...,1,V113091029
8,Provisioning attribute value UFF000007009635 i...,1,V113062392
9,Provisioning attribute value UFF000007009674 i...,1,V113062384


## 5. Orders on a fallback (unmatched) plan

Successfully created, but on `STATIC_FALLBACK_PLAN` rather than a matched
plan — worth reviewing before calling the migration done.

In [20]:
if df_order_results is not None and not df_order_results.empty:
    unmatched_plans = df_order_results[
        (df_order_results["status"] == "success") & (df_order_results["plan_matched"] == False)  # noqa: E712
    ]
    print(f"{len(unmatched_plans):,} successfully-created orders used the STATIC_FALLBACK_PLAN")
else:
    unmatched_plans = pd.DataFrame()

unmatched_plans


0 successfully-created orders used the STATIC_FALLBACK_PLAN


,SubscriptionUSN,TargetAccountNumber,status,plan_matched,productName,priceplanName,onebill_order_id,error


## 6. End-to-end funnel

In [21]:
_df_subs = try_load_df("subscriptions_resolved")
funnel = {
    "subscriptions loaded":     len(_df_subs) if _df_subs is not None else 0,
    "addresses created":        (df_address_results["status"] == "created").sum() if df_address_results is not None else 0,
    "orders attempted":         len(df_order_results) if df_order_results is not None else 0,
    "orders succeeded":         (df_order_results["status"] == "success").sum() if df_order_results is not None else 0,
}
pd.Series(funnel, name="count").to_frame()


,count
subscriptions loaded,323
addresses created,130
orders attempted,256
orders succeeded,246
